## Search Notebooks for Lab Results & Clinic Activity

In [ ]:
import sys
sys.path.append('..')
from credentials import *

from elasticsearch_utils import *

import duckdb
import pandas as pd
import numpy as np
import re

data_path = "../data/"
raw_data_path = data_path+'raw_data/'

### Connect to Elastic

In [ ]:
# --- Connect to Elasticsearch ---
es = connect_elasticsearch(hosts=hosts, username=username, password=password, api_key=True)

## Load Patients of Interest

In [ ]:
cols = ['master_person_id', 'patient_identifier3', 'patient_identifier4', 'patient_identifier2']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "chronic_kidney_disease_refined_inclusion_patients.csv"))[cols]

In [ ]:
id_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier3']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            try:
                id_number.append(int(i))
            except:
                pass

In [ ]:
nhs_number_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[1].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            nhs_number_dict[nhs_num]=row[0]

In [ ]:
#gstt_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier4']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(i)

In [ ]:
gstt_num_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[2].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            gstt_num_dict[nhs_num]=row[0]

In [ ]:
#epic_number = []
for idx, row in enumerate(inclusion_patients_df['patient_identifier2']):
    for i in row.split("'"):
        if len(i) <= 2:
            pass
        elif 'none' in i.lower():
            pass
        elif 'linked' in i.lower():
            pass
        else:
            id_number.append(int(i))

In [ ]:
epic_mrn_dict = {}
for idx, row in enumerate(inclusion_patients_df.values):
    for nhs_num in row[3].split("'"):
        if len(nhs_num) <= 2:
            pass
        elif 'none' in nhs_num.lower():
            pass
        elif 'linked' in nhs_num.lower():
            pass
        else:
            epic_mrn_dict[nhs_num]=row[0]

## Activity Data

### Inpatient:
- No. of AKI episodes
- No. A&E attendances
- No. ICU admissions
- (No. of hospitalisation days prior to start of dialysis)

### Outpatient:
- Missed / DNA outpatient appointments as of total in listed specialties
- No. of nephrology visits prior to dialysis
- Time to starting dialysis since first nephrology contact 

## A&E Activity

In [ ]:
# A&E Inpatient Stays
with duckdb.connect() as conn:
    ae_omop_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                              SELECT DISTINCT voi.master_person_id
                                     , voi.encounter_id AS activity_identifier
                                     , 'Legacy' AS activity_source
                                     , voi.visit_start_date AS activity_date
                                     , 'A&E' AS activity_type
                              FROM int_visit_occurrence_inptspell AS voi
                                  INNER JOIN inclusion_patients_df AS ip
                                      USING(master_person_id)
                              WHERE UPPER(voi.admit_specialty_source_value) LIKE '%A&E%' OR UPPER(voi.admit_specialty_source_value) LIKE '%ACCIDENT%' OR
                                    UPPER(voi.discharge_specialty_source_value) LIKE '%A&E%' OR UPPER(voi.discharge_specialty_source_value) LIKE '%ACCIDENT%' OR
                                    UPPER(voi.treatment_function_source_value) LIKE '%A&E%' OR UPPER(voi.treatment_function_source_value) LIKE '%ACCIDENT%' OR
                                    UPPER(voi.main_specialty_source_value) LIKE '%A&E%' OR UPPER(voi.main_specialty_source_value) LIKE '%ACCIDENT%'
                              ORDER BY voi.master_person_id, voi.visit_start_datetime;""").df()

In [ ]:
ae_omop_df['activity_date'] = pd.to_datetime(ae_omop_df['activity_date']).dt.date

ae_omop_df.head()

In [ ]:
index = 'notes'

columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'activity_identifier1', 'activity_Date',
           'activity_Type', 'activity_Department', 'activity_DepartmentSpecialty', 'activity_HospitalService']

In [ ]:
n = 65536

In [ ]:
ae_epic_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    epic_ae_query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "activity_DepartmentSpecialty : \"Emergency\" AND activity_Type : \"Hospital Encounter\"",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                { "terms": { "patient_activity_document_identifiers": sub_list }}
                                ],
                            "minimum_should_match": 1
                            }
                        },
                    {
                        "range": {
                            "activity_Date": {
                                "gte": "2023-10-01T00:00:00.000+01:00"
                                }
                            }
                        }
                    ]
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=epic_ae_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(ae_epic_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        ae_epic_df = pd.concat([ae_epic_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {ae_epic_df.shape[0]:,}\n")

    i+=1

In [ ]:
ae_epic_df.head()

In [ ]:
cols = ['master_person_id', 'activity_identifier1', 'activity_source', 'activity_date', 'activity_type']

ae_epic_df['masterPersonId3'] = ae_epic_df['patient_identifier3'].map(nhs_number_dict)
ae_epic_df['masterPersonId4'] = ae_epic_df['patient_identifier4'].map(gstt_num_dict)
ae_epic_df['masterPersonId2'] = ae_epic_df['patient_identifier2'].map(epic_mrn_dict)

ae_epic_df.insert(0, 'master_person_id', ae_epic_df['masterPersonId3'].combine_first(ae_epic_df['masterPersonId4']).combine_first(ae_epic_df['masterPersonId2']))

ae_epic_df = ae_epic_df[ae_epic_df['activity_HospitalService'].isin(['Emergency', np.NaN])]

ae_epic_df['activity_date'] = pd.to_datetime(ae_epic_df['activity_Date']).dt.date
ae_epic_df['activity_type'] = 'A&E'
ae_epic_df['activity_source'] = 'EPIC'

ae_epic_df = ae_epic_df[cols].drop_duplicates()

ae_epic_df.columns = ['master_person_id', 'activity_identifier', 'activity_source', 'activity_date', 'activity_type']

ae_epic_df.head()

In [ ]:
emergenies_adm_df = pd.concat([ae_omop_df, ae_epic_df])

print(f'Number of Legacy/OMOP Results: {ae_omop_df.shape[0]:,}')
print(f'Number of EPIC Results: {ae_epic_df.shape[0]:,}')

del ae_omop_df, ae_epic_df

emergenies_adm_df = emergenies_adm_df.sort_values(by=['master_person_id', 'activity_date']).reset_index(drop=True)

print(f'Total Number of Results: {emergenies_adm_df.shape[0]:,}')

emergenies_adm_df.head()

### Export Data

In [ ]:
# --- Save Results ---
file_name = "20251202_emergency_activity_search_results.csv"

emergenies_adm_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")

## Intensive Care Stays

In [ ]:
index = 'notes'

columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'activity_identifier1', 'activity_Date',
           'activity_Type', 'activity_Department', 'activity_DepartmentSpecialty', 'activity_HospitalService']

In [ ]:
n = 65536

In [ ]:
ic_epic_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    epic_ic_query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "activity_Department : (\"Ward List\")",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                { "terms": { "patient_activity_document_identifiers": sub_list }}
                                ],
                            "minimum_should_match": 1
                            }
                        },
                    {
                        "range": {
                            "activity_Date": {
                                "gte": "2023-10-01T00:00:00.000+01:00"
                                }
                            }
                        }
                    ]
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=epic_ic_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(ic_epic_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        ic_epic_df = pd.concat([ic_epic_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {ic_epic_df.shape[0]:,}\n")

    i+=1

In [ ]:
cols = ['master_person_id', 'activity_identifier1', 'activity_source', 'activity_date', 'activity_type']

ic_epic_df['masterPersonId3'] = ic_epic_df['patient_identifier3'].map(nhs_number_dict)
ic_epic_df['masterPersonId4'] = ic_epic_df['patient_identifier4'].map(gstt_num_dict)
ic_epic_df['masterPersonId2'] = ic_epic_df['patient_identifier2'].map(epic_mrn_dict)

ic_epic_df.insert(0, 'master_person_id', ic_epic_df['masterPersonId3'].combine_first(ic_epic_df['masterPersonId4']).combine_first(ic_epic_df['masterPersonId2']))

ic_epic_df['activity_date'] = pd.to_datetime(ic_epic_df['activity_Date']).dt.date
ic_epic_df['activity_type'] = 'IC'
ic_epic_df['activity_source'] = 'EPIC'

ic_epic_refined_df = ic_epic_df[cols].drop_duplicates()

del ic_epic_df

ic_epic_refined_df.columns = ['master_person_id', 'activity_identifier', 'activity_source', 'activity_date', 'activity_type']

ic_epic_refined_df.head()

In [ ]:
index = 'all'

columns = ['patient_identifier3', 'patient_identifier4', 'patient_identifier2', 'activity_VisitNumber', 'activity_Date', 'activity_VisitCurrentLocation']

In [ ]:
n = 65536

In [ ]:
ic_legacy_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    legacy_ic_query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "(activity_VisitCurrentLocation : \"Hospital Location\")",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                { "terms": { "patient_identifier1": sub_list }},
                                { "terms": { "patient_identifier3": sub_list }},
                                { "terms": { "patient_identifier2": sub_list }}
                                ],
                            "minimum_should_match": 1
                            }
                        }
                    ]
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=legacy_ic_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(ic_legacy_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        ic_legacy_df = pd.concat([ic_legacy_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {ic_legacy_df.shape[0]:,}\n")

    i+=1

In [ ]:
cols = ['master_person_id', 'activity_VisitNumber', 'activity_source', 'activity_date', 'activity_type']

ic_legacy_df['masterPersonId3'] = ic_legacy_df['patient_identifier3'].map(nhs_number_dict)
ic_legacy_df['masterPersonId4'] = ic_legacy_df['patient_identifier4'].map(gstt_num_dict)
ic_legacy_df['masterPersonId2'] = ic_legacy_df['patient_identifier2'].map(epic_mrn_dict)

ic_legacy_df.insert(0, 'master_person_id', ic_legacy_df['masterPersonId3'].combine_first(ic_legacy_df['masterPersonId4']).combine_first(ic_legacy_df['masterPersonId2']))

ic_legacy_df['activity_date'] = pd.to_datetime(ic_legacy_df['activity_Date']).dt.date
ic_legacy_df['activity_type'] = 'IC'
ic_legacy_df['activity_source'] = 'Legacy'

ic_legacy_refined_df = ic_legacy_df[cols].drop_duplicates()

del ic_legacy_df

ic_legacy_refined_df.columns = ['master_person_id', 'activity_identifier', 'activity_source', 'activity_date', 'activity_type']

ic_legacy_refined_df.head()

In [ ]:
intensive_care_activity_df = pd.concat([ic_legacy_refined_df, ic_epic_refined_df])

print(f'Number of Legacy/OMOP Results: {ic_legacy_refined_df.shape[0]:,}')
print(f'Number of EPIC Results: {ic_epic_refined_df.shape[0]:,}')

del ic_legacy_refined_df, ic_epic_refined_df

intensive_care_activity_df = intensive_care_activity_df.sort_values(by=['master_person_id', 'activity_date']).reset_index(drop=True)

print(f'Total Number of Results: {intensive_care_activity_df.shape[0]:,}')

intensive_care_activity_df.head()

In [ ]:
intensive_care_activity_df['master_person_id'].nunique()

### Export Results

In [ ]:
# --- Save Results ---
file_name = "20251202_intensive_care_activity_search_results.csv"

intensive_care_activity_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")

## Outpatient Appointments

In [ ]:
index = 'noting'

columns = ['patient_identifier4', 'patient_identifier2', 'patient_identifier3', 'activity_VisitSpecialtyCode', 'activity_VisitAdmissionType', 'activity_VistType',
           'activity_VisitSpecialtyName', 'activity_Date', 'activity_VisitNumber', 'document_OrderName', 'document_Name', 'document_CreatedWhen', 'document_Content']

In [ ]:
n = 65536

In [ ]:
op_legacy_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    legacy_op_query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "activity_VisitSpecialtyCode : (10281 OR 10741 OR 10751 OR 10761 OR 10781 OR 10782 OR 13045 OR 13051 OR 13061 OR 13081 OR 19143 OR 30731 OR 30741 OR 30742 OR 30751 OR 30761 OR 30769 OR 30771 OR 30781 OR 30789 OR 32041 OR 32043 OR 32044 OR 32045 OR 32046 OR 32047 OR 32051 OR 32053 OR 32055 OR 32056 OR 32057 OR 32061 OR 32063 OR 32065 OR 32066 OR 32067 OR 32071 OR 32073 OR 32075 OR 32076 OR 32077 OR 32081 OR 32082 OR 32083 OR 32084 OR 32085 OR 32086 OR 32087 OR 32861 OR 32881 OR 36132 OR 36142 OR 36151 OR 36152 OR 36161 OR 36164 OR 36171 OR 36172 OR 36181 OR 36182 OR 36183 OR 36184 OR 40041 OR 40043 OR 40045 OR 40052 OR 40053 OR 40061 OR 40062 OR 40081 OR 43041 OR 43042 OR 43043 OR 43044 OR 43051 OR 43061 OR 43081 OR 43082 OR 46081 OR 65051 OR 65351 OR 66341 OR 66351) AND activity_VisitAdmissionType : (\"Attended\" OR \"Did Not Attend\" OR \"Cancelled\" OR \"Arrived late - seen\" OR \"Cancelled by Hospital\" OR \"Cancelled by Patient\" OR \"Arrived late - not seen\" OR \"Cancelled by patient\")",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                { "terms": { "patient_identifier1": sub_list }},
                                { "terms": { "patient_identifier3": sub_list }},
                                { "terms": { "patient_identifier2": sub_list }}
                                ],
                            "minimum_should_match": 1
                            }
                        }
                    ]
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=legacy_op_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(op_legacy_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        op_legacy_df = pd.concat([op_legacy_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {op_legacy_df.shape[0]:,}\n")

    i+=1

In [ ]:
cols = ['master_person_id', 'activity_VisitNumber', 'activity_VisitSpecialtyName', 'activity_VisitAdmissionType', 'activity_source', 'activity_date', 'activity_type']

op_legacy_df['masterPersonId3'] = op_legacy_df['patient_identifier3'].map(nhs_number_dict)
op_legacy_df['masterPersonId4'] = op_legacy_df['patient_identifier4'].map(gstt_num_dict)
op_legacy_df['masterPersonId2'] = op_legacy_df['patient_identifier2'].map(epic_mrn_dict)

try:
    op_legacy_df.insert(0, 'master_person_id', op_legacy_df['masterPersonId3'].combine_first(op_legacy_df['masterPersonId4']).combine_first(op_legacy_df['masterPersonId2']))
except:
    pass

op_legacy_df['activity_date'] = pd.to_datetime(op_legacy_df['activity_Date'].apply(lambda x: str(x)[:10]), format='mixed').dt.date
op_legacy_df['activity_type'] = 'OP'
op_legacy_df['activity_source'] = 'Legacy'

op_legacy_refined_df = op_legacy_df[cols].drop_duplicates()

#del op_legacy_df

op_legacy_refined_df.columns = ['master_person_id', 'activity_identifier', 'activity_clinic', 'activity_visitOutcome', 'activity_source', 'activity_date', 'activity_type']

op_legacy_refined_df.head()

In [ ]:
index = 'notes'

columns = ['patient_identifier3', 'patient_identifier2', 'patient_EpicId', 'activity_identifier1', 'activity_Date', 'activity_DepartmentKey', 'activity_Department',
           'activity_DepartmentSpecialty', 'activity_DepartmentSpecialtyAbbreviation', 'activity_Type', 'activity_VisitClass', 'activity_PatientClass',
          ]

In [ ]:
n = 65536

In [ ]:
op_epic_df = pd.DataFrame()
i = 1
results = 0

full_list = list_chunker(id_number, n=n)
total_chunks = len(full_list)

for sub_list in full_list:
    epic_op_query = {
        "from": 0,
        "size": 10000,
        "query": {
            "bool": {
                "must": [
                    {
                        "query_string": {
                            "query": "activity_PatientClass : \"Outpatient\" AND activity_DepartmentKey : (\"4156\" OR \"7309\" OR \"3088\" OR \"20269\" OR \"29515\" OR \"22402\" OR \"9444\" OR \"16987\" OR \"3739\" OR \"10916\" OR \"6211\" OR \"4816\" OR \"7418\" OR \"6838\" OR \"9959\" OR \"7410\" OR \"21525\" OR \"17976\" OR \"2610\" OR \"19644\" OR \"8584\" OR \"26755\" OR \"19835\" OR \"12213\" OR \"1848\" OR \"26574\" OR \"27368\" OR \"15265\" OR \"29587\" OR \"27431\" OR \"2659\" OR \"13873\" OR \"10782\" OR \"22998\" OR \"28306\" OR \"11010\" OR \"21301\" OR \"6499\" OR \"266\" OR \"30264\" OR \"291\" OR \"29663\" OR \"23400\" OR \"21044\" OR \"13587\" OR \"12684\" OR \"12793\" OR \"2915\" OR \"24946\" OR \"1319\" OR \"26644\" OR \"9439\" OR \"8405\" OR \"30458\" OR \"1422\" OR \"8318\" OR \"20418\" OR \"5459\" OR \"21789\" OR \"30780\" OR \"23686\" OR \"7717\" OR \"13534\" OR \"13224\" OR \"785\" OR \"16953\" OR \"29825\" OR \"3855\" OR \"8664\" OR \"23374\" OR \"30867\" OR \"14976\" OR \"10736\" OR \"21252\" OR \"14933\" OR \"27026\" OR \"20420\" OR \"4270\" OR \"24918\" OR \"9127\" OR \"28348\" OR \"29355\" OR \"26656\" OR \"3087\" OR \"30353\" OR \"615\" OR \"7529\" OR \"25203\" OR \"30630\" OR \"20679\" OR \"1566\" OR \"10608\" OR \"29220\" OR \"16217\" OR \"569\" OR \"29258\" OR \"15828\" OR \"18992\" OR \"21691\" OR \"5550\" OR \"27736\" OR \"11724\" OR \"4682\" OR \"23137\" OR \"6334\" OR \"22960\" OR \"20111\" OR \"17837\" OR \"5237\" OR \"1026\" OR \"20417\" OR \"9426\" OR \"23477\" OR \"15906\" OR \"24759\" OR \"24680\" OR \"14264\" OR \"8174\" OR \"23088\" OR \"862\" OR \"22669\" OR \"30739\" OR \"24113\" OR \"8827\" OR \"24848\" OR \"26354\" OR \"30269\" OR \"5604\" OR \"29988\" OR \"26377\" OR \"8814\" OR \"16017\" OR \"24878\" OR \"3264\" OR \"6867\" OR \"16520\" OR \"9544\" OR \"5779\" OR \"3439\" OR \"15792\" OR \"2568\" OR \"28707\" OR \"20249\" OR \"16912\" OR \"26881\" OR \"14162\" OR \"7768\" OR \"22128\" OR \"25899\" OR \"21754\" OR \"18164\" OR \"3618\" OR \"13019\" OR \"19107\" OR \"7695\" OR \"10870\" OR \"4688\" OR \"30403\" OR \"27593\" OR \"12073\" OR \"10812\" OR \"723\" OR \"81868\" OR \"14051\" OR \"22345\" OR \"6852\" OR \"78892\" OR \"53693\" OR \"79330\" OR \"7240\" OR \"15113\" OR \"31342\" OR \"18708\" OR \"9032\" OR \"15447\" OR \"10808\" OR \"30766\" OR \"13940\" OR \"6962\" OR \"53696\" OR \"27429\" OR \"13565\" OR \"9351\" OR \"82075\" OR \"69010\" OR \"10194\" OR \"86233\" OR \"16482\" OR \"10265\" OR \"14019\" OR \"49972\" OR \"12027\" OR \"53695\" OR \"78888\" OR \"31653\" OR \"25339\" OR \"12976\" OR \"49875\" OR \"31923\" OR \"15856\" OR \"104622\" OR \"19585\" OR \"53670\" OR \"25182\" OR \"8444\" OR \"82514\" OR \"25128\" OR \"21041\" OR \"71490\" OR \"31587\" OR \"7507\" OR \"106091\" OR \"106170\" OR \"105721\" OR \"106859\" OR \"24364\" OR \"10844\" OR \"84531\" OR \"83096\" OR \"82695\" OR \"107039\" OR \"106721\" OR \"78907\" OR \"106885\" OR \"28790\" OR \"9766\" OR \"105400\" OR \"106057\" OR \"106698\" OR \"5320\" OR \"6489\")",
                            "analyze_wildcard": True,
                            "time_zone": "Europe/London"
                            }
                        }
                    ],
                "filter": [
                    {
                        "bool": {
                            "should": [
                                { "terms": { "patient_identifier1": sub_list }},
                                { "terms": { "patient_identifier3": sub_list }},
                                { "terms": { "patient_identifier2": sub_list }}
                                ],
                            "minimum_should_match": 1
                            }
                        }
                    ]
                }
            }
        }

    temp_df = es_docs_to_df(es, index=index, query=epic_op_query, column_headers=columns, timeout=600)

    if not temp_df.empty:
        try:
            results += temp_df.shape[0]
            temp_df = temp_df[~temp_df['_id'].isin(op_epic_df['_id'].to_list())]
        except Exception as e:
            print(f"Deduplication error: {e}")

        op_epic_df = pd.concat([op_epic_df, temp_df], ignore_index=True)

    del temp_df
    print(f'{(i/total_chunks)*100:.2f}% Complete ({i}/{total_chunks})')
    print(f"Total Documents Found:             {results:,}")
    print(f"Total Relevant & Unique Documents: {op_epic_df.shape[0]:,}\n")

    i+=1

In [ ]:
cols = ['master_person_id', 'activity_identifier1', 'activity_DepartmentKey', 'activity_Department', 'activity_DepartmentSpecialty', 'activity_source', 'activity_date', 'activity_type']

op_epic_df['masterPersonId3'] = op_epic_df['patient_identifier3'].map(nhs_number_dict)
op_epic_df['masterPersonId2'] = op_epic_df['patient_identifier2'].map(epic_mrn_dict)

try:
    op_epic_df.insert(0, 'master_person_id', op_epic_df['masterPersonId3'].combine_first(op_epic_df['masterPersonId2']))
except:
    pass

op_epic_df['activity_date'] = pd.to_datetime(op_epic_df['activity_Date'].apply(lambda x: str(x)[:10]), format='mixed').dt.date
op_epic_df['activity_type'] = 'OP'
op_epic_df['activity_source'] = 'EPIC'

op_epic_refined_df = op_epic_df[cols].drop_duplicates()

In [ ]:
op_legacy_refined_df = pd.read_csv(os.path.join(raw_data_path, "elasticsearch_search_hits", "OP_Legacy.csv"))

op_legacy_refined_df.head()

In [ ]:
op_epic_refined_df = pd.read_csv(os.path.join(raw_data_path, 'elasticsearch_search_hits', 'full_op_epic_data.csv'))

op_epic_refined_df['activity_identifier1'] = op_epic_refined_df['activity_identifier1'].astype(str)
op_epic_refined_df['activity_DepartmentKey'] = op_epic_refined_df['activity_DepartmentKey'].astype(str)
op_epic_refined_df['activity_Department'] = op_epic_refined_df['activity_Department'].astype(str)

op_epic_refined_df.head()

In [ ]:
activity_data_df = pd.read_csv(os.path.join(raw_data_path, 'elasticsearch_search_hits', 'complete_epic_op_data.csv'))
activity_data_df['PatientEncounterId'] = activity_data_df['PatientEncounterId'].astype(str)

activity_data_df.columns = ['activity_identifier1', 'activity_visitOutcome', 'comparison_Department']

activity_data_df.head()

In [ ]:
addl_activity_data_df = pd.read_csv(os.path.join(raw_data_path, 'elasticsearch_search_hits', 'EPIC_OP_Activity_2.csv'))
addl_activity_data_df['PatientEncounterID'] = addl_activity_data_df['PatientEncounterID'].astype(str)

addl_activity_data_df.columns = ['activity_identifier1', 'activity_visitOutcome', 'comparison_Department']

addl_activity_data_df.head()

In [ ]:
activity_data_df = pd.concat([activity_data_df, addl_activity_data_df])

In [ ]:
cols = ['master_person_id', 'activity_identifier1', 'activity_clinic', 'activity_visitOutcome', 'activity_source', 'activity_date', 'activity_type']

op_epic_refined_df = op_epic_refined_df.merge(activity_data_df, how='left', on='activity_identifier1')

dept_filter = (op_epic_refined_df['activity_Department']==op_epic_refined_df['comparison_Department'])
outcome_filter = (op_epic_refined_df['activity_visitOutcome']!='Booked')

op_epic_refined_df = op_epic_refined_df[dept_filter&outcome_filter].drop_duplicates().reset_index(drop=True)
op_epic_refined_df['activity_clinic'] = op_epic_refined_df['activity_DepartmentKey'] + '-' + op_epic_refined_df['activity_Department']

op_epic_refined_df = op_epic_refined_df[cols].drop_duplicates()

op_epic_refined_df.columns = ['master_person_id', 'activity_identifier', 'activity_clinic', 'activity_visitOutcome', 'activity_source', 'activity_date', 'activity_type']

op_epic_refined_df.head()

### Combined OP Data

In [ ]:
op_activity_df = pd.concat([op_legacy_refined_df, op_epic_refined_df])

op_activity_df = op_activity_df.sort_values(by=['master_person_id', 'activity_date']).reset_index(drop=True)

#### Clean Visit Outcome

In [ ]:
def outcome_cleaning(outcome):
    if 'cancelled' in outcome.lower():
        return 'Cancelled'
    elif 'completed' in outcome.lower():
        return 'Attended'
    elif 'patient arrived too late to be seen' in outcome.lower():
        return 'Arrived late - not seen'
    else:
        return outcome

In [ ]:
op_activity_df.insert(3, 'activity_visitOutcome_Clean', op_activity_df['activity_visitOutcome'].apply(lambda x: outcome_cleaning(x)))

op_activity_df = op_activity_df.drop(columns=['activity_visitOutcome']).rename(columns={'activity_visitOutcome_Clean': 'activity_visitOutcome'})

op_activity_df.head()

### Export Data

In [ ]:
# --- Save Results ---
file_name = "20251203_outpatient_clinic_activity_search_results.csv"

op_activity_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")

### Populate Mising Activiy Clinic Name

In [ ]:
op_appt_df = pd.read_csv(os.path.join(raw_data_path, "20251203_outpatient_clinic_activity_search_results.csv"))

op_appt_df.head()

In [ ]:
missing_clinic_name_df = op_appt_df[op_appt_df['activity_clinic'].isna()].reset_index(drop=True)

activity_visitnumber_list = list(missing_clinic_name_df['activity_identifier'].unique())

print(f'Number of OP Activity Visits: {len(activity_visitnumber_list):,}')

In [ ]:
index = 'noting'

columns = ['activity_VisitNumber', 'activity_VisitSpecialtyCode']

In [ ]:
op_query = {
    "from": 0,
    "size": 10000,
    "query": {
        "bool": {
            "must": [],
            "filter": [
                {
                    "bool": {
                        "should": [
                            { "terms": { "activity_VisitNumber": activity_visitnumber_list }}
                            ],
                        "minimum_should_match": 1
                        }
                    }
                ]
            }
        }
    }

In [ ]:
op_visit_details_df = es_docs_to_df(es, index=index, query=op_query, column_headers=columns, timeout=600)

In [ ]:
op_visit_details_df = op_visit_details_df[['activity_VisitNumber', 'activity_VisitSpecialtyCode']].drop_duplicates().reset_index(drop=True)

op_visit_details_df['activity_VisitSpecialtyCode'] = op_visit_details_df['activity_VisitSpecialtyCode'].astype(int)

print(f'Numer of OP Appointments: {op_visit_details_df.shape[0]:,}')

In [ ]:
specialty_codes_index_df = pd.read_csv(os.path.join(data_path, 'specialty_codes_master_index.csv'))[['Spec Code', 'Specialty Name']]

specialty_codes_index_df.columns = ['activity_VisitSpecialtyCode', 'activity_clinic']

specialty_codes_index_df['activity_VisitSpecialtyCode'] = specialty_codes_index_df['activity_VisitSpecialtyCode'].astype(int)

In [ ]:
op_visit_populated_details_df = op_visit_details_df.merge(specialty_codes_index_df, how='left', on='activity_VisitSpecialtyCode')

try:
    op_visit_populated_details_df['activity_clinic_name'] = op_visit_populated_details_df.apply(lambda row: str(row['activity_VisitSpecialtyCode'])+'-'+row['activity_clinic'], axis=1)
except:
    op_visit_populated_details_df['activity_clinic_name'] = ''

op_visit_populated_details_df = op_visit_populated_details_df[['activity_VisitNumber', 'activity_clinic_name']]

In [ ]:
op_appt_df = op_appt_df.merge(op_visit_populated_details_df, how='left', left_on='activity_identifier', right_on='activity_VisitNumber')

op_appt_df['activity_clinic'] = op_appt_df['activity_clinic'].combine_first(op_appt_df['activity_clinic_name'])

op_appt_df = op_appt_df.drop(columns=['activity_VisitNumber', 'activity_clinic_name'])

op_appt_df.head()

#### Export Repopulated Data

In [ ]:
# --- Save Results ---
file_name = "20251203_outpatient_clinic_activity_search_results.csv"

op_appt_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")

## AKI Activity

In [ ]:
with duckdb.connect() as conn:
    aki_activity_df = conn.execute("""ATTACH 'omop.duckdb' as PROD; USE PROD;
                                            SELECT DISTINCT co.master_person_id
                                                   , co.master_condition_occurrence_id
                                                   , co.master_visit_occurrence_id
                                                   , c2.concept_name AS visit_concept_name
                                                   , vo.appointment_id
                                                   , vo.encounter_id
                                                   , c1.concept_name AS condition_concept_name
                                                   , co.condition_source_value AS condition_icd10_code
                                                   , co.condition_start_date
                                                   , CASE WHEN UPPER(co.source_table_provenance) LIKE '%EPIC%' THEN 'EPIC'
                                                          WHEN UPPER(c2.concept_name) LIKE '%INPATIENT%' THEN 'Inpatient'
                                                          ELSE 'Outpatient' END AS activity_identifier
                                            FROM ext_condition_occurrence AS co
                                                INNER JOIN inclusion_patients_df AS ip
                                                    ON co.master_person_id = ip.master_person_id
                                                LEFT JOIN concept AS c1
                                                    ON co.condition_concept_id = c1.concept_id
                                                LEFT JOIN ext_visit_occurrence AS vo
                                                    ON co.master_visit_occurrence_id = vo.master_visit_occurrence_id
                                                LEFT JOIN concept AS c2
                                                    ON vo.visit_concept_id = c2.concept_id
                                            WHERE UPPER(co.condition_source_value) LIKE 'N17%'
                                            ORDER BY co.master_person_id, co.condition_start_date;""").df()

In [ ]:
aki_refined_activity_df = aki_activity_df.groupby('master_visit_occurrence_id').first().reset_index()

### Export Data

In [ ]:
# --- Save Results ---
file_name = "20251203_aki_activity_search_results.csv"

aki_refined_activity_df.to_csv(f"{raw_data_path}/{file_name}", index=False)
print("✅ Results saved.")

## Sandbox